# Auswertung Zelasto Studie

![Bild Depth Touch](images/depth-touch.jpg)

Jupyter Notebook um die in der Zelasto Studie geschriebenen CSV Dateien auf und auszuwerten. 

1. [Studienablauf](#Studienablauf)
1. [Aktuelle Lösung](Aktuelle-Lösung)
1. [Struktur Logfiles](#Struktur-Logfiles)
1. [Werte im Logfile](#Werte-im-Logfile)
1. [Statusabfolge](#Statusabfolge)
1. [Definition der Tiefenebenen](#Definition-der-Tiefenebenen)

## Studienablauf
---

* 6 Probe-Aufgaben (TASK) mit jeweils 2 Wiederholungen (TRIAL)
* 3 Blöcke (BLOCK) mit jeweils
    * 18 Aufgaben (TASK) mit jeweils 5 Wiederholungen (TRIAL)
    * In dem Block wurde immer eine Mapping-Methode (MAPPING_METHOD) genutzt
        * `direct / densening / widening`
    * Variiert wurden in den Aufgaben (TASK) die Anzahl der Tiefenebenen:
        * `6 / 9 / 12 / 15 / 18 / 21`
    * Variiert in den Wiederholungen (TRIALS) wurden die Zielebenen (random)
* Zuordnung Ebene <-> MAPPING_METHOD <-> Tiefe in eigener csv (depthlayers.csv)
* Erfolgs-/Fehlermöglichkeiten:
    * `COMPLETED / FAILED / TERMINATED`

## Aktuelle Lösung
---

* Iterativer / objektorientierter Ansatz (nicht wirklich Data Science strukturiert)
* Gesucht: besser anpassbare, „einfache“ data science Lösung

## Struktur Logfiles
---

`2021-05-11T14:55:08.419Z; INTERACTION; direct;18;14,9,5,1,17;18;1; 0.5099127;0.5482645;-0.449999869;637563489084131200;1;8`

| DateTime (LogServer)     | STATE          | mapping method | Task-No. | Target Layers | Layer-Count | Trial-Idx | PosX             | PosY                                                  | PosZ         | TimeStamp (Server) | InteractionType | CurrentLayer |
| ------------------------ | -------------- | -------------- | -------- | ------------- | ----------- | --------- | ---------------- | ----------------------------------------------------- | ------------ | ------------------ | --------------- | ------------ |
| 2021-05-11T14:55:08.419Z | INTERACTION    | direct         | 18       | 14,9,5,1,17   | 18          | 1         | 0.5099127        | 0.5482645                                             | -0.449999869 | 637563489084131200 | 1               | 8            |
| 2021-05-10T12:14:37.259Z | VIEW           | direct         | 2        | 7,8,1,5,2     | 9           | 0         | TASK_DESCRIPTION |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.259Z | TASK           | direct         | 2        | 7,8,1,5,2     | 9           | 0         |                  |                                                       |              |                    |                 |              |
| 2021-05-10T12:13:51.730Z | MAPPING_METHOD | direct         | 1        | 4,5,3,1,2     | 6           | 0         |                  |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.264Z | SUBTASK        | direct         | 2        | 7,8,1,5,2     | 9           | 0         | 2                |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.265Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | START            |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:39.587Z | VIEW           | direct         | 2        | 7,8,1,5,2     | 9           | 0         | TASK_VIEW        |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:45.064Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | HOLD             |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:46.565Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | COMPLETED        |                                                       |              |                    |                 |              |
| 2021-05-10T12:10:39.115Z | SUBTASK_STATE  | direct         | 1        | 1,5           | 6           | 0         | FAILED           | wrong level approved                                  |              |                    |                 |              |
| 2021-05-10T12:13:38.243Z | SUBTASK_STATE  | densening      | 6        | 3,12          | 14          | 0         | TERMINATED       | switched to other level before hold timeout completed |              |                    |                 |              |

## Werte im Logfile
---

* Tasks:
    * 1-6 (Test)
    * 1-18 (Block 1)
    * 19-36 (Block 2)
    * 37 - 54 (Block 3)

* 3 Blocks for any mapping method

* STATE:
  
    | State Value        | Description                                       | SubTypes           |
    | ------------------ | ------------------------------------------------- | ------------------ |
    | __INTERACTION__    | Trial interaction                                 | -                  |
    | __VIEW__           | switched view (describe Task)                     | TASK_VIEW          |
    |                    | switched view in test runs (describe Task)        | Test Run TASK_VIEW |
    |                    |                                                   | TASK_DESCRIPTION   |
    | __TASK__           |                                                   |                    |
    | __MAPPING_METHOD__ | starting to next large block (mapping method)     |                    |
    | __SUBTASK__        | same as the next: starting to next task           |                    |
    | __SUBTASK_STATE__  | start of the trial after Subtask                  | START              |
    |                    | dwell time in a layer exceeded: hold-timer starts | HOLD               |
    |                    | end of the trial (Success)                        | COMPLETED          |
    |                    | end of the trial (hold failure                    | TERMINATED         |
    |                    | end of the trial (wrong level ?)                  | FAILED             |

* mapping method - describes, how layers are aligned:
    * __direct__ (equivalent distance)
    * __densening__ (larger distance on top, decreasing with depth value)
    * __widening__ (narrower distance on top, increasing with depth value)

* Task-No.
    * running number of tasks

* Target Layers, Trial-Idx
    * array of targets for each layer in current Task
    * trial-Index (zero based) describes target layer for current trial

* Layer-Count
    * number of max layers in Task

* PosX, PosY, PosZ
    * Positions received from Tracking Server
    * PosX / PosY in range [0, 1]
    * PosZ in range [-1, 1] with 0 = on the surface / no interaction, -1 max push, +1 max pull

* Timestamp (Server)
    * timestamp received from Tracking server: miliseconds based unix time stamp

* InteractionType
    * type recognized from server (1 = PUSH)

* current layer
    * layer associated with received depth value

## Statusabfolge
---

1. START (missing on first trial !) OR
    * start of an new task
2. TASK_VIEW / TASK_DESCRIPTION
3. HOLD
4. COMPLETED / TERMINATED / FAILED
   

## Definition der Tiefenebenen
---

__Location:__ `data/depthlayers.csv` [File](data/depthlayers.csv)

`6; direct; 0 | 1 | 2 | 3 | 4 | 5 | 6; 0 | 0.0834 | 0.25 | 0.4167 | 0.5834 | 0.75 | 0.9167 | 1`

| NUM_LAYERS       | MAPPING_METHOD            | DEPTH_LAYER_ID                         | DEPTH_LAYER_BORDER              |
| ---------------- | ------------------------- | -------------------------------------- | ------------------------------- |
| number of layers | mapping method for  block | idx of the depth layer in border array | start depths for each layer idx |

* mapping of layers to depth ranges for better evaluation how "good" a depth layer has been hit


## Aufgaben
---

### Vorverarbeitung - Cleaning

* Probe-Aufgaben und eigentliche Studie in verschiedene Dateien
* Ersetzen: Pro Task, muss im ersten Trial  TASK_VIEW mit START getauscht werden
* Spalten benennen (für bessere Adressierung)
* Nebenbedingung/Bonus: sowohl „nur Studie“ / „nur Test“ / „Test und Studie“ zusammen laden  Ordner-Struktur / Namensschema ?
* Lösung/Code dokumentieren (jupyter notebook)


In [45]:
# csv Dateien sind im Verzeichnis ../data zu finden

import pandas as pd
import glob

path = "../data/"
all_files = glob.glob(path + "/*.csv")

# for debugging: only use first file
all_files = all_files[9:11]

study =[]
cleanedStudy = []
columnNames =["DateTime", "State", "mappingMethod", "TaskNo", "TargetLayers", "LayerCount", "TrialIdx", "posX", "posY", "posZ", "TimeStamp", "iteractionType", "currentLayer"]
#testData = pd.read_csv("../data/01.csv", sep=";", names = columnNames) #load csv data with delimeter and add column names ;

totalLength = 0

for filename in all_files:
    df =pd.read_csv(filename, sep=";", names = columnNames)
    study.append(df)
    totalLength += len(df)
    
print("totallines: " + str(totalLength))

currentLine = 0
idx = 0

marker = []
marker.append(["Id", "EndInit", "EndTests", "SwapLines"])

for testData in study:
    endInit=0
    endTest=0
    
    idx += 1
    currentLine += len(testData)

    # drop data from the start and end of the file, that are not needed
    for i in range(1, len(testData)-2):                                        #iterate through the file to find the lines were the actual test starts ans ends
        if (testData["TaskNo"][i] == 6) and (testData["TaskNo"][i+1] == 1):
            endInit = i+1
        if (testData["TaskNo"][i] == 54) and (testData["TaskNo"][i+1] == 1):
            endTests = i+1

    marker.append([idx, endInit, endTests, "-"]);
    
    print("Processed: " + str(currentLine) + " of " +  str(totalLength) + "(" + str(idx) + ")")
    
    filteredData = testData.drop(testData.index[endTests:len(testData)])     
    filteredData = filteredData.drop(filteredData.index[0:endInit])

    filteredData.reset_index(drop=True, inplace=True)

    # exchange TASK_VIEW ans START in the first Trial for each Task
    startLine =0
    taskLine=0

    # iterate over all Lines to find START and TASK_VIEW 
    # Swap lines when TASK_VIEW was found
    for i in range(0, len(filteredData)-2):
        if filteredData["TrialIdx"][i] == 0 and filteredData["posX"][i] == " START":
            startLine = i
        if filteredData["TrialIdx"][i] == 0 and filteredData["posX"][i] == " TASK_VIEW":
            taskLine = i
            linecopy = filteredData.iloc[startLine]
            filteredData.iloc[startLine] = filteredData.iloc[taskLine]
            filteredData.iloc[taskLine] = linecopy
            
            marker.append([idx, "-", "-", str(startLine + endInit) + " <-> " + str(taskLine + endInit)]);

    cleanedStudy.append(filteredData)
    
markerframe = pd.DataFrame(data=marker)
markerframe.to_csv(r'marker.csv', sep= ";")
    
print('finished')

totallines: 206832
Processed: 115615 of 206832(1)
Processed: 206832 of 206832(2)
finished


### Extraktion

* Erfolg/Fehler für jeden Trial und je Proband

| PROBAND | BLOCK | TASK | TRIAL | MAPPING_METHOD | NUM_LAYERS | TARGET | TARGET_RELATIVE | RESULT    |
| ------- | ----- | ---- | ----- | -------------- | ---------- | ------ | --------------- | --------- |
| 1       | 1     | 2    | 5     | direct         | 15         | 7      | 0.46666         | COMPLETED |
| .       | .     | .    | .     | .              | .          | .      | .               | .         |
| .       | .     | .    | .     | .              | .          | .      | .               | .         |


In [50]:
import math
#create new Dataframe for the results per Trial and Proband
cn_result = ["Proband", "Block", "Task", "Trial", "MappingMethod", "NumLayers", "Target", "Target_Relative", "Result"]
endOfTrials = pd.DataFrame(columns = cn_result) 
probandNum = 0

# iterate through all probands
for proband in cleanedStudy:
    last_start = len(proband)
    
    # search for all results for each proband   
    for i in range (0, len(proband)):
        
        # to prevent "double results": note index of last_start
        if proband["posX"][i] == " START":
            last_start = i
        
        if last_start < i and (proband["posX"][i] == " COMPLETED" or proband["posX"][i] == " FAILED" or proband["posX"][i] == " TERMINATED"):
            # add Result to Dataframe
            # copy get the needed values and addd entry to the dataframe
            blockNo = math.ceil(proband["TaskNo"][i] / 18)
            taskNo = proband["TaskNo"][i] - ((blockNo - 1) * 18)
            layerCount = proband["LayerCount"][i]
            target = int(proband["TargetLayers"][i].split(",")[proband["TrialIdx"][i]])
                        
            endOfTrials.loc[len(endOfTrials)] = [probandNum, blockNo, taskNo, proband["TrialIdx"][i], proband["mappingMethod"][i], layerCount, target, target / layerCount, proband["posX"][i] ]
            
            last_start = len(proband)
            
    probandNum = probandNum+1
    print("Processed: " + str(probandNum) + " of " + str(len(cleanedStudy)))
display(endOfTrials)

endOfTrials.to_csv(r'results.csv', sep= ";")



Processed: 1 of 2
Processed: 2 of 2


,Proband,Block,Task,Trial,MappingMethod,NumLayers,Target,Target_Relative,Result
0,0,1,1,0,densening,15,11,0.733333,COMPLETED
1,0,1,1,1,densening,15,14,0.933333,COMPLETED
2,0,1,1,2,densening,15,8,0.533333,COMPLETED
3,0,1,1,3,densening,15,1,0.066667,TERMINATED
4,0,1,1,4,densening,15,4,0.266667,COMPLETED
...,...,...,...,...,...,...,...,...,...
535,1,3,18,0,direct,9,1,0.111111,COMPLETED
536,1,3,18,1,direct,9,5,0.555556,TERMINATED
537,1,3,18,2,direct,9,2,0.222222,TERMINATED
538,1,3,18,3,direct,9,8,0.888889,COMPLETED



* Versuchsdauer für jeden Trial und je Proband

| PROBAND | BLOCK | TASK | TRIAL | MAPPING_METHOD | NUM_LAYERS | TARGET | DURATION |
| ------- | ----- | ---- | ----- | -------------- | ---------- | ------ | ------   |
| 1       | 1     | 2    | 5     | direct         | 15         | 7      | 15.532s  |
| .       | .     | .    | .     | .              | .          | .      | .        |
| .       | .     | .    | .     | .              | .          | .      | .        |


In [49]:
from datetime import datetime, timedelta

#create new Dataframe for the results per Trial and Proband
cn_times = ["Proband", "Block", "Task", "Trial", "MappingMethod", "NumLayers", "Target", "Target_Relative", "Duration"]
end_times = pd.DataFrame(columns = cn_times) 
probandNum = 0

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

# iterate through all probands
for proband in cleanedStudy:
    last_start = len(proband)
    time_start = datetime(1,1,1)

    # search for all results for each proband   
    for i in range (0, len(proband)):       
        
        # to prevent "double results": note index of last_start
        # save starting time
        if proband["posX"][i] == " START":
            last_start = i
            time_start = datetime.strptime(proband["DateTime"][i], timeFormat)
            
        if last_start < i and (proband["posX"][i] == " COMPLETED" or proband["posX"][i] == " FAILED" or proband["posX"][i] == " TERMINATED"):
            # add Result to Dataframe
            # copy get the needed values and addd entry to the dataframe
            blockNo = math.ceil(proband["TaskNo"][i] / 18)
            taskNo = proband["TaskNo"][i] - ((blockNo - 1) * 18)
            layerCount = proband["LayerCount"][i]
            target = int(proband["TargetLayers"][i].split(",")[proband["TrialIdx"][i]])
            time_end = datetime.strptime(proband["DateTime"][i], timeFormat)

            delta = (time_end - time_start).total_seconds()
            
            end_times.loc[len(end_times)] = [probandNum, blockNo, taskNo, proband["TrialIdx"][i], proband["mappingMethod"][i], layerCount, target, target / layerCount, delta ]
            
            time_start = datetime(1,1,1)
            time_end = datetime(1,1,1)            
            last_start = len(proband)

    probandNum = probandNum+1
    print("Processed: " + str(probandNum) + " of " + str(len(cleanedStudy)))
display(end_times)

end_times.to_csv(r'times.csv', sep= ";")

Processed: 1 of 2
Processed: 2 of 2


,Proband,Block,Task,Trial,MappingMethod,NumLayers,Target,Target_Relative,Duration
0,0,1,1,0,densening,15,11,0.733333,10.926
1,0,1,1,1,densening,15,14,0.933333,9.491
2,0,1,1,2,densening,15,8,0.533333,9.525
3,0,1,1,3,densening,15,1,0.066667,20.049
4,0,1,1,4,densening,15,4,0.266667,6.846
...,...,...,...,...,...,...,...,...,...
535,1,3,18,0,direct,9,1,0.111111,6.573
536,1,3,18,1,direct,9,5,0.555556,2.958
537,1,3,18,2,direct,9,2,0.222222,3.536
538,1,3,18,3,direct,9,8,0.888889,13.482


* Erfolg/Fehlerquote über alle Probanden je Ebenenanzahl und mapping methode

| MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED  | TERMINATED | SUM |
| ---            | ---        | ---       | ---     | ---        | --- |
| direct         | 5          | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .              | .          | .         | .       | .          | .   |
| .              | .          | .         | .       | .          | .   |

In [209]:
df_resultStat = pd.read_csv('results.csv', sep=";", names = cn_result)
df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["MappingMethod", "NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["MappingMethod", "NumLayers", "Result"]) 

groups = df_resultStat.groupby(["MappingMethod", "NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0][0], elem[0][1], 0, 0, 0, 0, 0, 0, 0]

    
count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["MappingMethod"] == index[0]) & (df_rates["NumLayers"] == index[1])]
    r_index = rate.index
        
    if index[2] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 2] = compl_sum
        
    if index[2] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 4] = fail_sum
        
    if index[2] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 6] = term_sum
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates)


,MappingMethod,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
0,densening,12,9,0.3,1,0.033333,20,0.666667,30,1.0
1,densening,15,18,0.6,0,0.0,12,0.4,30,1.0
2,densening,18,15,0.5,1,0.033333,14,0.466667,30,1.0
3,densening,21,12,0.4,4,0.133333,14,0.466667,30,1.0
4,densening,6,8,0.266667,1,0.033333,21,0.7,30,1.0
5,densening,9,9,0.3,4,0.133333,17,0.566667,30,1.0
6,direct,12,13,0.433333,1,0.033333,16,0.533333,30,1.0
7,direct,15,13,0.433333,1,0.033333,16,0.533333,30,1.0
8,direct,18,19,0.633333,3,0.1,8,0.266667,30,1.0
9,direct,21,14,0.466667,0,0.0,16,0.533333,30,1.0


* Plot: über alle Probanden für eine bestimmte Konstellation aus mapping method, Ebenenanzahl, Zielebene 
    * Y: Ebenen / Tiefe
    * X: Zeit
* Bonus: Y-Achse nach realen Tiefenwerten  skaliert (depthlayers.csv)

| PROBAND | TIMESTAMP | CURRENT_LAYER | Z     |
| ---     | ---       | ---           | ---   |
| 154     | 12345678  | 6             | -0.54 |
| .       | .         | .             | .     |
| .       | .         | .             | .     |

### Bonus
---
* depthlayers.csv: Mapping der Tiefenwerte auf die zugehörige Ebene nach Mapping-Methode Achtung: Z-Werte in Studie sind im Bereich [–1,0] zu bewerten, depthlayers.csv gibt die absolut-Werte an!
* Für erfolgreiche Trials: Jeweils zwischen HOLD und COMPLETED 
    * Maximale Schwankung (% der Ebenenbreite)
    * Durchschnittstiefe (prozentualer Abstand zur Ebenenmitte) 
    * Wie „weit“ waren die Probanden von einem Fehlschlag entfernt / wie sicher haben Sie die ebene gehalten ?
* „Wackler“ vor HOLD (wie oft wurde die Zielebene erreicht, aber nicht gehalten ?)
* Dies jeweils nach Ebenenanzahl und Mapping_Method gruppiert (über alle Probanden)
* Statistische Auswertung – Vorschläge?
* Hypothesen:
    * Sweet Spot für Anzahl der Ebenen (bspw: ab wann nimmt die Fehlerquote plötzlich stark zu ? Oder: bei welcher Ebenenanzahl wird eine Genauigkeit > 95% im Schnitt erzielt ?)
    * Vergleich der Mapping_Method: 
        * H1: widening ist genauer als direct und densening - mit zunehmender Tiefe nimmt die benötigte Kraft zu  Genauigkeit sinkt, deswegen sind die „unteren“ (aka höheren) Ebenen weiter auseinander
        * H2: densening ist genauer als direct und widening - mit zunehmender Tiefe nimmt die benötigte Kraft zu  konstante Kraftnutzung für den Ebenenwechsel = „untere“ (aka höhere) Ebenen näher beieinander
        * H3: direct ist genauer als widening und densening - Kraft spielt keine (oder nur geringfügige) Rolle, wichtig ist die äquidistante Positionierung der Tiefenebenen


In [ ]:
# your code here



In [ ]:
print(test)